# RAG Demo

In [1]:
# handle asynchronous operations within a Jupyter Notebook environment
import nest_asyncio
nest_asyncio.apply() 

In [2]:
import requests

# The /dashboard path returns HTML webpage text, NOT JSON data structure.
response = requests.get("http://localhost:6333/livez")

#  GOOD: Check the status code instead of trying to parse JSON data
if response.status_code == 200:
    print("✅ Qdrant running!")
else:
    print(f"❌ Qdrant not running: {response.status_code}")

✅ Qdrant running!


In [3]:
import qdrant_client

collection_name="chat_with_docs"

client = qdrant_client.QdrantClient(
    host="localhost",
    port=6333
)



In [4]:
# load text from docs and put into memory

from llama_index.core import SimpleDirectoryReader

input_dir_path = './docs'

loader = SimpleDirectoryReader(
            input_dir = input_dir_path,
            required_exts=[".pdf"],
            recursive=True
        )
docs = loader.load_data()

type(docs), len(docs)

(list, 32)

In [5]:
# create index to later user for querying similar items.

from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

def create_index(documents):
    vector_store = QdrantVectorStore(client=client, collection_name=collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
    
    return index

In [6]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5",trust_remote_code=True)

Settings.embed_model = embed_model

index = create_index(docs)


/Users/thetsajeet/Documents/workspace/labs/rag_labs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-07 10:42:36,296 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-07 10:42:36,298 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-07 10:42:36,313 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json?%2FBAAI%2Fbge-large-en-v1.5%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-07 10:42:36,549 - INFO - HTTP Request: HEAD https://huggingf

In [7]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3.2:1b", request_timeout=120.0)

Settings.llm = llm

In [8]:
from llama_index.core import PromptTemplate

template = """Context information is below:
              ---------------------
              {context_str}
              ---------------------
              Given the context information above I want you to think
              step by step to answer the query in a crisp manner,
              incase you don't know the answer say 'I don't know!'
            
              Query: {query_str}
        
              Answer:"""

qa_prompt_tmpl = PromptTemplate(template)


In [9]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2", 
    top_n=3
)


2026-10-07 10:42:45,125 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-07 10:42:45,431 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-10-07 10:42:45,436 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-2-v2, initializing a new CrossEncoder model.
2026-10-07 10:42:45,738 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-2-v2 "HTTP/1.1 307 Temporary Redirect"
2026-10-07 10:42:46,045 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L2-v2 "HTTP/1.1 200 OK"
2026-10-07 10:42:46,352 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-2-v2 "HTTP/1.1 307 Temporary Redirect"
2026-10-07 10:42:46,660 - INFO - HTTP Request: GET https://huggingface.co/api/m

In [10]:
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[rerank])

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt_tmpl}
)

response = query_engine.query("What exactly is DSPy?")


2026-10-07 10:42:49,927 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-10-07 10:42:49,989 - INFO - HTTP Request: POST http://localhost:6333/collections/chat_with_docs/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 91.79it/s]
2026-10-07 10:42:50,598 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


In [11]:
from IPython.display import Markdown, display

display(Markdown(str(response)))

Based on the provided context information, DSPy stands for "Descriptive Signatures for Predictive Models".